# 🚦 Indian Traffic Object Detection with YOLOv8

**Beginner-friendly Jupyter Notebook classroom demo**

Upload a traffic photograph and detect common objects such as cars, motorcycles, buses, trucks, bicycles, pedestrians, traffic lights and stop signs.

- Runs locally on a **CPU**.
- Uses the small `yolov8n.pt` pretrained model.
- The first model run downloads its weights, so an internet connection is needed once.
- No VisionPort account or API is required.

> **Important:** The general COCO model does not have a dedicated auto-rickshaw class. It may miss or misclassify auto-rickshaws. We can add India-specific training in a later stage.


## 1. Install the required packages
Run this cell once in your notebook.

In [ ]:
%pip install -q ultralytics ipywidgets matplotlib pandas pillow

## 2. Import libraries and load YOLOv8

We use the **nano** model (`yolov8n.pt`) because it is lightweight and suitable for CPU demonstrations.

In [ ]:
from ultralytics import YOLO
from PIL import Image
from IPython.display import display
import ipywidgets as widgets
import matplotlib.pyplot as plt
import pandas as pd
from collections import Counter
import io
import numpy as np

# Load the pretrained model. The weights download automatically on first use.
model = YOLO("yolov8n.pt")

print("YOLOv8 model loaded successfully.")
print("Inference device: CPU")

## 3. Upload an Indian traffic photograph

Run this cell, click **Choose Files**, and select a JPG, JPEG or PNG traffic image from your computer.

In [ ]:
uploader = widgets.FileUpload(
    accept=".jpg,.jpeg,.png,.webp",
    multiple=False,
    description="Choose traffic photo"
)
display(uploader)

## 4. Read the uploaded photograph

After selecting the image above, run this cell. It handles both common `ipywidgets` upload formats.

In [ ]:
def get_uploaded_file(upload_widget):
    value = upload_widget.value

    if not value:
        raise ValueError("No image uploaded. Run the upload cell and choose an image first.")

    # ipywidgets 8 generally returns a tuple of file dictionaries.
    if isinstance(value, dict):
        # Some ipywidgets versions return a dictionary keyed by filename.
        if "content" in value:
            file_info = value
        else:
            file_info = next(iter(value.values()))
    else:
        file_info = value[0]

    content = file_info["content"]
    image_bytes = bytes(content)
    filename = file_info.get("name", "uploaded_traffic_image")
    return Image.open(io.BytesIO(image_bytes)).convert("RGB"), filename

traffic_image, image_filename = get_uploaded_file(uploader)

print("Uploaded:", image_filename)
print("Image size:", traffic_image.size)

plt.figure(figsize=(10, 6))
plt.imshow(traffic_image)
plt.axis("off")
plt.title("Original traffic photograph")
plt.show()

## 5. Run object detection

Change the confidence threshold if required. A lower threshold may find more objects but can also produce false detections.

In [ ]:
CONFIDENCE_THRESHOLD = 0.25  # Try 0.40 or 0.50 for stricter detections
IMAGE_SIZE = 640             # Try 416 for faster CPU inference

results = model.predict(
    source=traffic_image,
    conf=CONFIDENCE_THRESHOLD,
    imgsz=IMAGE_SIZE,
    device="cpu",
    verbose=False
)

result = results[0]
print("Detection completed.")

## 6. Display the detected image
Bounding boxes show where the model found each object. Labels show the predicted class and confidence.

In [ ]:
# Ultralytics result.plot() returns a BGR image; convert it to RGB for Matplotlib.
annotated_bgr = result.plot()
annotated_rgb = annotated_bgr[:, :, ::-1]

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
axes[0].imshow(traffic_image)
axes[0].set_title("Original image")
axes[0].axis("off")

axes[1].imshow(annotated_rgb)
axes[1].set_title("YOLOv8 object-detection result")
axes[1].axis("off")

plt.tight_layout()
plt.show()

## 7. Show object names, confidence scores and bounding-box coordinates

In [ ]:
detection_rows = []

if result.boxes is not None:
    for box in result.boxes:
        class_id = int(box.cls.item())
        detection_rows.append({
            "Object": result.names[class_id],
            "Confidence": round(float(box.conf.item()), 3),
            "X-min": round(float(box.xyxy[0][0].item()), 1),
            "Y-min": round(float(box.xyxy[0][1].item()), 1),
            "X-max": round(float(box.xyxy[0][2].item()), 1),
            "Y-max": round(float(box.xyxy[0][3].item()), 1),
        })

detections_df = pd.DataFrame(
    detection_rows,
    columns=["Object", "Confidence", "X-min", "Y-min", "X-max", "Y-max"]
)

if detections_df.empty:
    print("No objects detected. Try a clearer image or lower the confidence threshold.")
else:
    display(detections_df)

## 8. Count detected objects by category

In [ ]:
if detections_df.empty:
    print("No objects to count.")
else:
    object_counts = detections_df["Object"].value_counts().rename_axis("Object").reset_index(name="Count")
    display(object_counts)

    print("Total objects detected:", len(detections_df))
    print("Different object categories:", detections_df["Object"].nunique())

## 9. Save the annotated image and detection table

In [ ]:
# Save the annotated image in the current notebook folder.
annotated_image = Image.fromarray(annotated_rgb)
annotated_image.save("traffic_detection_result.jpg")

# Save detection details as a CSV file.
detections_df.to_csv("traffic_detection_results.csv", index=False)

print("Saved: traffic_detection_result.jpg")
print("Saved: traffic_detection_results.csv")

## 10. Classroom discussion and experiments

Ask students to repeat the experiment with different images and compare:

1. Daylight versus night-time traffic.
2. Empty roads versus crowded junctions.
3. Confidence thresholds of 0.25, 0.40 and 0.60.
4. Image sizes of 320, 416 and 640.
5. Clear vehicles versus partly hidden (occluded) vehicles.

**Questions for students**
- Why might a motorcycle behind a bus be missed?
- What happens when the confidence threshold is increased?
- Why might the model confuse an auto-rickshaw with another vehicle class?
- How could we improve detection for Indian traffic?

**Limitations:** This is an educational prototype, not a traffic-enforcement or safety-critical system. COCO pretrained classes include person, bicycle, car, motorcycle, bus, truck, traffic light and stop sign, but not a dedicated auto-rickshaw category. Fine-tuning on a suitable Indian traffic dataset is a later project stage.

**Model note:** This notebook uses Ultralytics YOLOv8. Review its licensing terms before redistribution or commercial use: https://www.ultralytics.com/license
